# Modular Embedding Construction

Weighting all features equally causes melody n-grams to be overpower the other features since it accounts for the majority of dimensions. Therefore, alternative methods for weighting, scaling and normalising should be explored to give all features better representation in simmilairty comparison.

In [12]:
from pathlib import Path
from collections import defaultdict

from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances

import matplotlib.pyplot as plt


import numpy as np
import pandas as pd
from tqdm import tqdm
import pretty_midi

import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config.paths import MSD_METADATA_DIR, PROCESSED_DATA_DIR, LMD_MATCHED_DIR, NOTEBOOK_DATASETS_DIR

In [13]:
if Path(f"{PROCESSED_DATA_DIR}/simmilarity_features_30.parquet").exists():
    features_df = pd.read_parquet(f"{PROCESSED_DATA_DIR}/simmilarity_features_30.parquet")
else:
    print("Features file not found. Please run the feature extraction script first.")

## Produce Vector Embeddings

### Vocanularies for ngram features

In [14]:
from collections import Counter

def process_ngrams(col: str):

    counts = Counter()

    for ngrams in features_df[col]:
        counts.update(ngrams.keys())

    vocab = {
        ngram: i
        for i, (ngram, _) in enumerate(
            counts.most_common(500)
        )
    }
    return vocab, counts

melody_vocab, melody_counts = process_ngrams("melody_ngrams")
low_vocab, low_counts = process_ngrams("low_ngrams")
mid_vocab, mid_counts = process_ngrams("mid_ngrams")
high_vocab, high_counts = process_ngrams("high_ngrams")

In [15]:
def dict_to_counter(d):

    return Counter({
        tuple(map(int, k.split(","))): v
        for k, v in d.items()
    })

In [16]:
def vectorise_counter(counter, vocab):

    vec = np.zeros(len(vocab))
    
    total = sum(counter.values())

    if total == 0:
        return vec

    for item, count in counter.items():

        if item in vocab:
            vec[vocab[item]] = count / total

    return vec

In [17]:
def build_embedding(row, ngram_vocab, low_vocab, mid_vocab, high_vocab):

    parts = []

    stats = np.array([
        row["note_count"],
        row["mean_pitch"],
        row["std_pitch"],
        row["pitch_range"],
        row["mean_duration"],
        row["std_duration"],
        row["mean_velocity"],
        row["std_velocity"],
        row["mean_instruments"],
    ])

    # chroma
    chroma = np.array([
        row["total_chroma"],
        row["low_chroma"],
        row["mid_chroma"],
        row["high_chroma"],
        row["key_strength"],
        row["key_distance"],
    ])


    # entropy
    entropy = np.array([
        row["pitch_entropy"],
        row["duration_entropy"],
        row["ioi_entropy"]
    ])

    # melody (ngrams)
    melody = vectorise_counter(
        row["melody_ngrams"],
        ngram_vocab
    )

    # low, mid, high (ngrams)
    low = vectorise_counter(
    row["low_ngrams"],
    low_vocab
    )
    mid = vectorise_counter(
        row["mid_ngrams"],
        mid_vocab
    )
    high = vectorise_counter(
        row["high_ngrams"],
        high_vocab
    )


    # rhythm
    rhythm = np.array([
        row["ioi_hist"],
        row["duration_hist"],
        row["drum_ioi_hist"],
        row["bass_ioi_hist"],
        row["melody_ioi_hist"],
        row["density"],
        row["drum_density"],
        row["bass_density"],
        row["melody_density"]
    ])

    # structure
    structure = np.array([
        row["segment_chroma"],
        row["chord_progression"],
        row["harmonic_rhythm"],
        row["segment_variation"]
    ])

    return np.concatenate(parts)

In [18]:
import numpy as np

def build_embedding(
    row,
    ngram_vocab,
    low_vocab,
    mid_vocab,
    high_vocab,
    weights=None
):

    # Default weights (easy to tweak later)
    if weights is None:
        weights = {
            "stats": 1.0,
            "chroma": 1.0,
            "entropy": 0.5,
            "rhythm": 1.0,
            "structure": 0.5,
            "melody": 0.25,
            "low": 0.15,
            "mid": 0.15,
            "high": 0.15,
        }

    parts = []

    # -------------------------
    # Statistical features
    # -------------------------
    stats = np.array([
        row["note_count"],
        row["mean_pitch"],
        row["std_pitch"],
        row["pitch_range"],
        row["mean_duration"],
        row["std_duration"],
        row["mean_velocity"],
        row["std_velocity"],
        row["mean_instruments"],
    ], dtype=float)

    parts.append(stats * weights["stats"])

    # -------------------------
    # Chroma
    # -------------------------
    chroma = np.concatenate([
        np.asarray(row["total_chroma"], dtype=float),
        np.asarray(row["low_chroma"], dtype=float),
        np.asarray(row["mid_chroma"], dtype=float),
        np.asarray(row["high_chroma"], dtype=float),
        np.asarray([row["key_strength"]], dtype=float),
        np.asarray(row["key_distance"], dtype=float),
    ])

    parts.append(chroma * weights["chroma"])

    # -------------------------
    # Entropy
    # -------------------------
    entropy = np.array([
        row["pitch_entropy"],
        row["duration_entropy"],
        row["ioi_entropy"],
    ], dtype=float)

    parts.append(entropy * weights["entropy"])

    # -------------------------
    # Rhythm
    # -------------------------
    rhythm = np.concatenate([
        np.asarray(row["ioi_hist"], dtype=float),
        np.asarray(row["duration_hist"], dtype=float),
        np.asarray(row["drum_ioi_hist"], dtype=float),
        np.asarray(row["bass_ioi_hist"], dtype=float),
        np.asarray(row["melody_ioi_hist"], dtype=float),
        np.asarray(row["density"], dtype=float),
        np.asarray(row["drum_density"], dtype=float),
        np.asarray(row["bass_density"], dtype=float),
        np.asarray(row["melody_density"], dtype=float),
    ])

    parts.append(rhythm * weights["rhythm"])

    # -------------------------
    # Structure
    # -------------------------
    structure = np.concatenate([
        np.asarray(row["segment_chroma"], dtype=float).flatten(),
        np.asarray(row["chord_progression"], dtype=float),
        np.asarray([row["harmonic_rhythm"]], dtype=float),
        np.asarray([row["segment_variation"]], dtype=float),
    ])

    parts.append(structure * weights["structure"])

    # -------------------------
    # Melody n-grams
    # -------------------------
    melody = vectorise_counter(
        row["melody_ngrams"],
        ngram_vocab
    )

    parts.append(melody * weights["melody"])

    # -------------------------
    # Low register n-grams
    # -------------------------
    low = vectorise_counter(
        row["low_ngrams"],
        low_vocab
    )

    parts.append(low * weights["low"])

    # -------------------------
    # Mid register n-grams
    # -------------------------
    mid = vectorise_counter(
        row["mid_ngrams"],
        mid_vocab
    )

    parts.append(mid * weights["mid"])

    # -------------------------
    # High register n-grams
    # -------------------------
    high = vectorise_counter(
        row["high_ngrams"],
        high_vocab
    )

    parts.append(high * weights["high"])

    return np.concatenate(parts)

In [20]:
X = np.vstack([
    build_embedding(row, melody_vocab, low_vocab, mid_vocab, high_vocab)
    for _, row in features_df.iterrows()
])

from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(X)

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 1 dimension(s) and the array at index 5 has 0 dimension(s)